In [1]:
import os
import langchain
from dotenv import load_dotenv
load_dotenv()

os.environ["OPENAI-API_KEY"]=os.getenv("openai_api_key")
os.environ["GROQ_API_KEY"]=os.getenv("groq_api_key")
os.environ["GOOGLE_API_KEY"]=os.getenv("google_api_key")


Middleware
Middleware provides a way to more tightly control what happens inside the agent. Middleware is useful for the following:

Tracking agent behavior with logging, analytics, and debugging.
Transforming prompts, tool selection, and output formatting.
Adding retries, fallbacks, and early termination logic.
Applying rate limits, guardrails, and PII detection.



Summarization MiddleWare
Automatically summarize conversation history when approaching token limits, preserving recent messages while compressing older context. Summarization is useful for the following:

Long-running conversations that exceed context windows.
Multi-turn dialogues with extensive history.
Applications where preserving full conversation context matters.

In [2]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

### Messagebased summarization
agent=create_agent(
    model="groq:qwen/qwen3-32b",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:qwen/qwen3-32b",
            trigger=("messages",10),
            keep=("messages",4)
        )
    ]
)

In [3]:
### Run with thread id
config={"configurable":{"thread_id":"test-1"}}

In [5]:
# Alternative test data
questions = [
    "What is 2+2?",
    "What is 10*5?",
    "What is 100/4?",
    "What is 15-7?",
    "What is 3*3?",
    "What is 4*4?",
]

for q in questions:
    response=agent.invoke({"messages":[HumanMessage(content=q)]},config)
    print(f"Messages: {response}")
    print(f"Messages: {len(response['messages'])}")

Messages: {'messages': [HumanMessage(content="Here is a summary of the conversation to date:\n\nError generating summary: Error code: 429 - {'error': {'message': 'Rate limit reached for model `qwen/qwen3-32b` in organization `org_01k6wsvge3f0bantgnbg6jemx3` service tier `on_demand` on tokens per minute (TPM): Limit 6000, Used 2297, Requested 3716. Please try again in 130ms. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}", additional_kwargs={'lc_source': 'summarization'}, response_metadata={}, id='8a4f35db-b3df-46f9-87d9-f80827230198'), AIMessage(content='<think>\nOkay, the user is asking "What is 15-7?" Let me think. This is a straightforward subtraction problem. First, I need to calculate 15 minus 7. Let me verify.\n\nStarting with 15, subtracting 7. If I count back from 15: 15, 14, 13, 12, 11, 10, 9, 8. That\'s 7 steps, so the result is 8. Alternatively, using number bonds: 15 is 10 + 5. Sub